# Backtests par periode

Modifiez la cellule `parameters` puis executez toutes les cellules. Les donnees, graphiques et la synthese seront enregistres dans le dossier `backtest/` a la racine du depot.

In [ ]:
from pathlib import Path
import sys

# Trouve la racine du depot pour charger sa configuration et ses modules.
racine = Path.cwd().resolve()
while not (racine / "src" / "strategy.py").is_file() and racine != racine.parent:
    racine = racine.parent
if not (racine / "src" / "strategy.py").is_file():
    raise FileNotFoundError("Impossible de trouver src/strategy.py depuis le dossier courant.")
sys.path.insert(0, str(racine / "src"))

In [ ]:
from config import (
    BACKTEST_LONG_WINDOW,
    BACKTEST_PERIODS,
    BACKTEST_SHORT_WINDOW,
    BACKTEST_TICKER,
    BACKTEST_TRANSACTION_COST,
)

TICKER = BACKTEST_TICKER
MOYENNE_COURTE = BACKTEST_SHORT_WINDOW
MOYENNE_LONGUE = BACKTEST_LONG_WINDOW
COUT_TRANSACTION = BACKTEST_TRANSACTION_COST
PERIODES = BACKTEST_PERIODS

In [ ]:
from pathlib import Path

import pandas as pd

from backtest import courbes
from metrics import metriques
from strategy import backtest_moyennes

dossier_resultats = racine / "backtest"
dossier_resultats.mkdir(parents=True, exist_ok=True)
synthese = []

for periode in PERIODES:
    print(f"Backtest {TICKER} - {periode['nom']}")
    donnees = backtest_moyennes(
        ticker=TICKER,
        courte=MOYENNE_COURTE,
        longue=MOYENNE_LONGUE,
        cout=COUT_TRANSACTION,
        debut=periode["debut"],
        fin=periode["fin"],
    )

    donnees.to_csv(
        dossier_resultats / f"{TICKER}_{periode['nom']}_donnees.csv",
        index=True,
    )
    courbes(
        {
            "Strategie MM": donnees["strategie"],
            "Achat-conservation": donnees["rendement"],
        },
        titre=f"{TICKER} - {periode['nom']} - MM {MOYENNE_COURTE}/{MOYENNE_LONGUE}",
        dossier=dossier_resultats,
        nom_fichier=f"{TICKER}_{periode['nom']}.png",
    )

    ligne = {
        "periode": periode["nom"],
        "debut": periode["debut"],
        "fin": periode["fin"],
    }
    ligne.update({f"strategie_{cle}": valeur for cle, valeur in metriques(donnees["strategie"]).items()})
    ligne.update({f"achat_conservation_{cle}": valeur for cle, valeur in metriques(donnees["rendement"]).items()})
    synthese.append(ligne)

tableau_synthese = pd.DataFrame(synthese)
tableau_synthese.to_csv(dossier_resultats / "synthese.csv", index=False)
display(tableau_synthese)
print(f"Resultats enregistres dans : {dossier_resultats}")